# Avance 2 – Integración con la API de Yelp (Chicago)
Me conecto a la API de Yelp, busco restaurantes de Chicago para cada preferencia alimenticia de los clientes y comparo lo que piden los clientes (demanda) con lo que hay en Yelp (oferta).

**Antes de correr:** este notebook lee `clientes_chicago_limpio.csv`, que se genera al correr `Avance_EDA.ipynb`.

# Explicacion variables de entorno

Durante mi investigacion para guardar variables de entorno se encontro como en los servicios de AWS, existen servicios como AWS Secrets Manager y AWS Systems Manager Parameter Store que permiten almacenar y gestionar secretos de manera segura. 

Estos servicios proporcionan características como el cifrado de datos, la rotación automática de secretos y el control de acceso basado en roles, 
lo que ayuda a proteger la información sensible y a cumplir con las mejores prácticas de seguridad.

Por motivos del ejercicio y asegurar que el evaluador tenga las herramientas para evaluar mi trabajo correctamente, se deja quemado las variables de entorno a pesar de que es un mala practica.

Aprendizaje: Archivos.env, python-dotenv

%pip install pandas numpy seaborn matplotlib

In [ ]:
# ============================================================
# BLOQUE 1: CONEXIÓN A LA API DE YELP
# ============================================================

# Importamos las librerías que vamos a usar
import requests        # requests sirve para pedir información a una API por internet
import pandas as pd    # pandas para trabajar con tablas
import numpy as np     # numpy lo usamos para marcar datos vacíos (np.nan)
import time            # time para hacer pausas entre llamadas a la API


# Dirección de la API de Yelp para buscar negocios
api_url = "https://api.yelp.com/v3/businesses/search"

# API Key de la cuenta del curso (entregada por la clase)
apikey = "FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx"

# Los "headers" son como la identificación que mostramos al pedir los datos
# Yelp pide que la clave vaya con la palabra "Bearer" adelante
headers = {"Authorization": "Bearer " + apikey}

# Usamos Chicago porque es la ciudad que filtramos en el Avance 1
ciudad = "Chicago"

# Primera consulta: pedimos 50 restaurantes de la ciudad
params = {"term": "restaurants", "location": ciudad, "limit": 50}
response = requests.get(api_url, params=params, headers=headers)

# El código de respuesta nos dice si funcionó:
# 200 = todo bien | 401 = la clave está mal | 429 = se acabaron las llamadas del día
print("Código de respuesta:", response.status_code)

if response.status_code == 200:
    data = response.json()    # convertimos la respuesta (formato JSON) en un diccionario de Python
    print("Llaves del diccionario:", data.keys())    # las secciones que trae la respuesta
    print("Restaurantes registrados en", ciudad, ":", data["total"])
    print("Ejemplo:", data["businesses"][0]["name"])  # nombre del primer restaurante
else:
    print("Algo falló. Mensaje de Yelp:", response.text)   # Yelp explica qué pasó

Código de respuesta: 200
Llaves del diccionario: dict_keys(['businesses', 'total', 'region'])
Restaurantes registrados en Chicago : 7900
Ejemplo: The Purple Pig


In [2]:
# ============================================================
# BLOQUE 2: EXTRACCIÓN DE DATOS DE YELP
# ============================================================

# --- Paso 1: Definir qué vamos a buscar ---

# Para cada preferencia de los clientes, la palabra que usamos para buscar en Yelp (en inglés)
# "Otro" y "No especificado" no se buscan porque no corresponden a un tipo de comida concreto
busquedas = {
    "Carnes": "steakhouse",
    "Mariscos": "seafood",
    "Pescado": "fish",
    "Vegetariano": "vegetarian",
    "Vegano": "vegan",
}

lista_restaurantes = []   # aquí iremos guardando cada restaurante que encontremos


# --- Paso 2: Pedir los datos a Yelp ---

# Recorremos cada preferencia del diccionario
for preferencia, termino in busquedas.items():
    print("Buscando:", preferencia, "->", termino)

    # Yelp entrega máximo 50 restaurantes por llamada, entonces pedimos 4 "páginas":
    # offset es desde qué posición empezar (0, 50, 100, 150) -> hasta 200 restaurantes por tipo
    for offset in [0, 50, 100, 150]:
        parametros = {
            "location": ciudad,          # ciudad (definida en el Bloque 1)
            "term": termino,             # tipo de comida
            "categories": "restaurants", # solo restaurantes (no tiendas ni supermercados)
            "limit": 50,                 # cuántos resultados por llamada
            "offset": offset,            # desde qué posición
        }
        respuesta = requests.get(api_url, params=parametros, headers=headers)

        # Si Yelp responde con error, mostramos el mensaje y pasamos a la siguiente preferencia
        if respuesta.status_code != 200:
            print("  Error", respuesta.status_code, "->", respuesta.text)
            break

        datos = respuesta.json()   # convertimos la respuesta JSON en diccionario

        negocios = datos["businesses"]   # la lista de restaurantes de esta página

        # Si ya no vienen más restaurantes, dejamos de pedir páginas
        if len(negocios) == 0:
            break

        # De cada restaurante sacamos solo los campos que nos interesan
        for negocio in negocios:
            # Las categorías vienen en una lista; las unimos en un solo texto separado por comas
            categorias = ", ".join([c["title"] for c in negocio["categories"]])

            # .get() trae el dato si existe, y si no existe deja un vacío en vez de dar error
            # (por ejemplo, muchos restaurantes no tienen precio)
            lista_restaurantes.append({
                "preferencias_alimenticias": preferencia,
                "id_yelp": negocio["id"],
                "nombre": negocio["name"],
                "categorias": categorias,
                "calificacion": negocio.get("rating"),
                "num_resenas": negocio.get("review_count"),
                "precio": negocio.get("price"),
                "direccion": negocio["location"].get("address1"),
                "codigo_postal": negocio["location"].get("zip_code"),
                "latitud": negocio["coordinates"].get("latitude"),
                "longitud": negocio["coordinates"].get("longitude"),
            })

        # Pausa de medio segundo entre llamadas para no saturar la API
        time.sleep(0.5)

    print("  Acumulados en la lista:", len(lista_restaurantes))


# --- Paso 3: Convertir a tabla y limpiar ---

# Convertimos la lista de restaurantes en una tabla de pandas
df_yelp = pd.DataFrame(lista_restaurantes)
print("Tamaño:", df_yelp.shape)
display(df_yelp.head())

# Revisamos si un restaurante salió repetido dentro de la misma preferencia y los quitamos
# (que un restaurante salga en "Vegano" y en "Vegetariano" sí es normal y se deja)
print("Repetidos dentro de la misma preferencia:", df_yelp.duplicated(["preferencias_alimenticias", "id_yelp"]).sum())
df_yelp = df_yelp.drop_duplicates(["preferencias_alimenticias", "id_yelp"])

# Revisamos los datos vacíos
display(df_yelp.isnull().sum().to_frame("nulos"))

# El precio viene como "$", "$$", "$$$" o "$$$$"; contamos los símbolos para tenerlo como número (1 a 4)
df_yelp["nivel_precio"] = df_yelp["precio"].str.len()


# --- Paso 4: Guardar los datos ---

# POR QUÉ: así no tenemos que volver a llamar a la API (la llave es compartida y tiene límite diario)
df_yelp.to_csv("restaurantes_yelp_chicago.csv", index=False)
print("Archivo guardado: restaurantes_yelp_chicago.csv")

Buscando: Carnes -> steakhouse
  Acumulados en la lista: 200
Buscando: Mariscos -> seafood
  Acumulados en la lista: 400
Buscando: Pescado -> fish
  Acumulados en la lista: 600
Buscando: Vegetariano -> vegetarian
  Acumulados en la lista: 800
Buscando: Vegano -> vegan
  Acumulados en la lista: 1000
Tamaño: (1000, 11)


,preferencias_alimenticias,id_yelp,nombre,categorias,calificacion,num_resenas,precio,direccion,codigo_postal,latitud,longitud
0,Carnes,apyIymDFbUK7u-Y3V5kmkA,Bavette's Bar & Boeuf,"Steakhouses, Seafood, Bars",4.6,3575,$$$$,218 W Kinzie,60654,41.889295,-87.634893
1,Carnes,sNQglz_HoXgnUi1KTN3Iew,Gibsons Bar & Steakhouse,"Steakhouses, Bars, American",4.1,2905,$$$,1028 N Rush St,60611,41.901402,-87.628086
2,Carnes,VPJk-SEWSWS_nGoQvM-COw,Penumbra,"Wine Bars, Seafood, Steakhouses",4.8,1038,$$$,3309 W Fullerton Ave,60647,41.924459,-87.710898
3,Carnes,UK5Xb8xqUOUBi5vQu0bQXQ,Maple & Ash,"Steakhouses, Seafood, Breakfast & Brunch",4.2,2556,$$$$,8 W Maple St,60610,41.902113,-87.628695
4,Carnes,2tKMs0DbdxPSl2JeCJUEKw,"Joe's Seafood, Prime Steak & Stone Crab","Seafood, Steakhouses",4.4,3294,$$$$,60 E Grand Ave,60611,41.891975,-87.625638


Repetidos dentro de la misma preferencia: 0


,nulos
preferencias_alimenticias,0
id_yelp,0
nombre,0
categorias,0
calificacion,0
num_resenas,0
precio,242
direccion,0
codigo_postal,0
latitud,0


Archivo guardado: restaurantes_yelp_chicago.csv


In [3]:
# ============================================================
# BLOQUE 3: INTEGRACIÓN DE YELP CON LA BASE DE CLIENTES
# ============================================================

# --- Paso 1: Cargar las dos bases ---

# Base de clientes limpia que guardamos en el Avance 1 (debe estar en la misma carpeta)
df_clientes = pd.read_csv("clientes_chicago_limpio.csv")

# Base de restaurantes que guardamos en el Bloque 2
# POR QUÉ: leemos el CSV en vez de llamar otra vez a la API, así no gastamos llamadas
df_yelp = pd.read_csv("restaurantes_yelp_chicago.csv")

# Calificación 0 con 0 reseñas no es una mala nota, es que no tiene nota: la dejamos vacía
df_yelp.loc[df_yelp["num_resenas"] == 0, "calificacion"] = np.nan

# Un mismo restaurante puede salir en varias búsquedas (ej. Vegano y Vegetariano)
# Contamos cuántos restaurantes distintos hay en total
print("Filas en la base de Yelp:", len(df_yelp), "| Restaurantes distintos:", df_yelp["id_yelp"].nunique())


# --- Paso 2: Demanda = resumen de clientes por preferencia alimenticia ---

# Contamos cuántos clientes hay en cada preferencia y qué % representan
demanda = df_clientes.groupby("preferencias_alimenticias").agg(clientes=("id_persona", "count"))
demanda["pct_clientes"] = (demanda["clientes"] / len(df_clientes) * 100).round(1)

# Gasto promedio por comida, solo con clientes que sí comen afuera
# POR QUÉ: los clientes con 0 visitas tienen gasto 0, y si los dejamos bajan el promedio "por comida"
comen_afuera = df_clientes[df_clientes["promedio_gasto_comida"] > 0]
demanda["gasto_promedio_usd"] = comen_afuera.groupby("preferencias_alimenticias")["promedio_gasto_comida"].mean().round(1)


# --- Paso 3: Oferta = resumen de restaurantes de Yelp por preferencia ---

# Agrupamos los restaurantes por preferencia y calculamos promedios de calidad y precio
# restaurantes_analizados = sobre cuántos restaurantes se calcularon los promedios
# restaurantes_con_precio = sobre cuántos se calculó el precio (no todos tienen precio en Yelp)
oferta = df_yelp.groupby("preferencias_alimenticias").agg(
    restaurantes_analizados=("id_yelp", "count"),
    restaurantes_con_precio=("nivel_precio", "count"),
    calificacion_promedio=("calificacion", "mean"),
    resenas_promedio=("num_resenas", "mean"),
    nivel_precio_promedio=("nivel_precio", "mean")).round(1)


# --- Paso 4: Unir demanda y oferta ---

# merge une las dos tablas usando la columna que tienen en común: preferencias_alimenticias
# how="left" mantiene todas las preferencias de los clientes, aunque no tengan datos en Yelp
integrado = pd.merge(demanda, oferta, on="preferencias_alimenticias", how="left")

# Ordenamos de la preferencia con más clientes a la de menos
integrado = integrado.sort_values("clientes", ascending=False)

print("==== Demanda (clientes) vs oferta (Yelp) por preferencia alimenticia ====")
display(integrado)
# NOTA: "Otro" y "No especificado" salen vacíos en las columnas de Yelp porque no se buscaron


# --- Paso 5: Guardar la tabla integrada para el Avance 3 ---
integrado.to_csv("integrado_clientes_yelp_chicago.csv")
print("Archivo guardado: integrado_clientes_yelp_chicago.csv")

Filas en la base de Yelp: 1000 | Restaurantes distintos: 680
==== Demanda (clientes) vs oferta (Yelp) por preferencia alimenticia ====


,clientes,pct_clientes,gasto_promedio_usd,restaurantes_analizados,restaurantes_con_precio,calificacion_promedio,resenas_promedio,nivel_precio_promedio
preferencias_alimenticias,,,,,,,,
Carnes,1589,29.5,26.3,200.0,169.0,4.0,909.5,2.9
Vegetariano,1123,20.9,27.7,200.0,155.0,4.3,741.6,2.2
Mariscos,696,12.9,27.7,200.0,138.0,4.2,765.8,2.6
Otro,591,11.0,26.3,NaN,NaN,NaN,NaN,NaN
Vegano,571,10.6,26.4,200.0,152.0,4.3,776.1,2.1
Pescado,546,10.1,26.3,200.0,144.0,4.1,580.5,2.2
No especificado,268,5.0,27.7,NaN,NaN,NaN,NaN,NaN


Archivo guardado: integrado_clientes_yelp_chicago.csv


## Resultados – Avance 2
- **Conexión:** la API respondió con código 200 y reporta unos 7.900 restaurantes en Chicago.
- **Extracción:** 1.000 registros (200 por cada preferencia buscada), que corresponden a 680 restaurantes distintos. Un mismo restaurante puede salir en varias búsquedas, por ejemplo en Vegano y en Vegetariano.
- **Calidad del dato:** 242 restaurantes no tienen precio en Yelp, y 1 tenía calificación 0 con 0 reseñas, que se dejó vacía.
- **Integración:** Carnes es la preferencia más pedida (29,5%) y en Yelp tiene la peor calificación (4,0) y el precio más alto (2,9 de 4). Vegetariano y Vegano están mejor calificados (4,3) y son más baratos (2,2 y 2,1 de 4). El gasto por comida casi no cambia entre preferencias (26–28 USD).
- **Límites:** son solo los primeros 200 resultados por búsqueda, los datos de Yelp cambian con el tiempo y "Otro" y "No especificado" no se buscaron.
- El gráfico y el análisis de esta tabla están al final del Bloque 6 de `Avance_EDA.ipynb` (después de correr este notebook, volver a correr ese).